In [1]:
import json
from collections import Counter, defaultdict

# Load the vocabulary data
with open('vocabeo_words.json', 'r', encoding='utf-8') as f:
    words = json.load(f)

print(f"Total number of words: {len(words)}")
print()

# Count word types
word_types = Counter(word['word_type'] for word in words)
print("=== Word Types ===")
for wtype, count in word_types.most_common():
    print(f"  {wtype}: {count}")
print()

# Count levels (handle None values)
levels = Counter(word.get('level', 'Unknown') for word in words)
print("=== CEFR Levels ===")
for level, count in sorted(levels.items(), key=lambda x: (x[0] is None, x[0])):
    display_level = level if level is not None else 'Unknown'
    print(f"  {display_level}: {count}")
print()

# Frequency distribution
frequencies = Counter(word['frequency'] for word in words)
print("=== Frequency Distribution ===")
for freq, count in sorted(frequencies.items(), key=lambda x: int(x[0]) if x[0].isdigit() else x[0]):
    print(f"  {freq}: {count}")
print()

# Fields per word type
print("=== Fields Present by Word Type ===")
fields_by_type = defaultdict(set)
for word in words:
    wtype = word['word_type']
    for key in word.keys():
        fields_by_type[wtype].add(key)

for wtype in sorted(fields_by_type.keys()):
    print(f"\n{wtype}:")
    for field in sorted(fields_by_type[wtype]):
        print(f"  - {field}")

# Show sample entries for each word type
print("\n=== Sample Entries by Word Type ===")
seen_types = set()
for word in words:
    wtype = word['word_type']
    if wtype not in seen_types:
        seen_types.add(wtype)
        print(f"\n--- {wtype} ---")
        print(json.dumps(word, indent=2, ensure_ascii=False))
        if len(seen_types) >= 10:  # Limit output
            break

# Additional statistics
print("\n=== Additional Statistics ===")

# Words with conjugation data
words_with_conjugation = sum(1 for word in words if word.get('conjugation') is not None)
print(f"Words with conjugation data: {words_with_conjugation}")

# Average number of examples per word
avg_examples = sum(len(word.get('examples', [])) for word in words) / len(words)
print(f"Average examples per word: {avg_examples:.1f}")

# Average number of translations per word
avg_translations = sum(len(word.get('translations', [])) for word in words) / len(words)
print(f"Average translations per word: {avg_translations:.1f}")

# Words with additional_info
words_with_additional = sum(1 for word in words if word.get('additional_info'))
print(f"Words with additional_info: {words_with_additional}")

# Most common words (by frequency 5)
high_freq_words = [word['word'] for word in words if word['frequency'] == '5']
print(f"\nWords with highest frequency (5): {len(high_freq_words)}")
print(f"Examples: {', '.join(high_freq_words[:20])}")

Total number of words: 6215

=== Word Types ===
  Noun: 3177
  Verb: 1456
  Adjective: 990
  Adverb: 348
  Pronoun: 63
  Number: 62
  Preposition: 57
  Conjunction: 34
  Interjection: 26
  Article: 2

=== CEFR Levels ===
  A1: 827
  A2: 683
  B1: 1689
  Unknown: 3016

=== Frequency Distribution ===
  1: 57
  2: 710
  3: 3433
  4: 1762
  5: 253

=== Fields Present by Word Type ===

Adjective:
  - additional_info
  - conjugation
  - examples
  - frequency
  - level
  - translations
  - word
  - word_type

Adverb:
  - additional_info
  - conjugation
  - examples
  - frequency
  - level
  - translations
  - word
  - word_type

Article:
  - additional_info
  - conjugation
  - examples
  - frequency
  - level
  - translations
  - word
  - word_type

Conjunction:
  - additional_info
  - conjugation
  - examples
  - frequency
  - level
  - translations
  - word
  - word_type

Interjection:
  - additional_info
  - conjugation
  - examples
  - frequency
  - level
  - translations
  - word
  - wo

In [7]:
l = [word["word"] for word in words if word["word_type"] == "Conjunction"]
for i in l:
    print(i)

und
als
dass
um... zu
aber
oder
wenn
damit
denn
ohne
sowie
weil
ob
sondern
während
beziehungsweise
je... (desto/umso...)
obwohl
nachdem
bevor
sowohl... als auch
weder... noch
sodass
indem
entweder... oder
solang(e)
falls
sobald
zumal
desto
sofern
ehe
umso
soviel


In [ ]:
import json
import os
import time
from typing import List
from pydantic import BaseModel, Field
import requests

# Load the vocabulary data
with open('vocabeo_words.json', 'r', encoding='utf-8') as f:
    words = json.load(f)

# Filter only verbs
verbs = [word for word in words if word['word_type'] == 'Verb']
print(f"Total verbs: {len(verbs)}")

# Define Pydantic model for structured output
class VerbConjugation(BaseModel):
    infinitive: str = Field(description="The infinitive form of the verb")
    present_3rd_person: str = Field(description="Present tense 3rd person singular form")
    simple_past: str = Field(description="Simple past (Präteritum) form")
    participle: str = Field(description="Past participle (Partizip II) form")

class VerbConjugationList(BaseModel):
    verbs: List[VerbConjugation]

# Minimax API configuration
MINIMAX_API_KEY = ""  # Replace with your actual API key
MINIMAX_API_URL = "https://api.minimaxi.chat/v1/text/chatcompletion_v2"
MODEL = "MiniMax-M2.7"

def get_verb_conjugations(verb_batch: List[str]) -> List[VerbConjugation]:
    """
    Get conjugations for a batch of verbs using Minimax API.
    """
    verb_list = ", ".join(verb_batch)
    
    system_prompt = """You are a German language expert. For each verb provided, return the present 3rd person singular, simple past (Präteritum), and past participle (Partizip II) forms.

Respond in JSON format with a list of objects, each containing:
- infinitive: the original infinitive form
- present_3rd_person: present tense 3rd person singular (er/sie/es form)
- simple_past: simple past/Präteritum form
- participle: past participle/Partizip II form

# RETURN ONLY THE JSON DATA, DO NOT INCLUDE ANY EXPLANATIONS OR ADDITIONAL TEXT. ONLY RAW JSON, NO MARKDOWN, NO CODE BLOCKS, NO TEXT, NO EXPLANATIONS, NOTHING ELSE.

Example:
[
  {
    "infinitive": "gehen",
    "present_3rd_person": "geht",
    "simple_past": "ging",
    "participle": "gegangen"
  }
]"""
    
    user_prompt = f"For the following German verbs, provide their conjugations: {verb_list}"
    
    headers = {
        "Authorization": f"Bearer {MINIMAX_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "verb_conjugations",
                "schema": VerbConjugationList.model_json_schema()
            }
        }
    }
    
    try:
        response = requests.post(MINIMAX_API_URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()
        
        result = response.json()
        print(f"API response for batch: {result}")
        content = result['choices'][0]['message']['content']
        
        # Parse the JSON response
        conjugation_data = json.loads(content)
        conjugations = VerbConjugationList(verbs=conjugation_data)
        
        return conjugations.verbs
        
    except Exception as e:
        print(f"Error processing batch: {e}")
        return []

# Process verbs in batches
BATCH_SIZE = 50  # Adjust based on API limits and response quality
all_conjugations = []
failed_batches = []

# Extract just the verb words
verb_words = [verb['word'] for verb in verbs]

print(f"Processing {len(verb_words)} verbs in batches of {BATCH_SIZE}...")

for i in range(0, len(verb_words), BATCH_SIZE):
    batch = verb_words[i:i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    total_batches = (len(verb_words) + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"\nProcessing batch {batch_num}/{total_batches}: {batch}")
    
    conjugations = get_verb_conjugations(batch)
    
    if conjugations:
        all_conjugations.extend(conjugations)
        print(f"Successfully got {len(conjugations)} conjugations")
    else:
        failed_batches.append(batch)
        print(f"Failed to get conjugations for batch {batch_num}")
    
    # Rate limiting - be nice to the API
    if i + BATCH_SIZE < len(verb_words):
        time.sleep(1)

print(f"\n=== Summary ===")
print(f"Total verbs processed: {len(verb_words)}")
print(f"Successful conjugations: {len(all_conjugations)}")
print(f"Failed batches: {len(failed_batches)}")

# Display first few results
print("\n=== Sample Results ===")
for conj in all_conjugations[:5]:
    print(f"{conj.infinitive}: {conj.present_3rd_person} | {conj.simple_past} | {conj.participle}")

# Save results to file
output_file = 'verb_conjugations.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump([conj.model_dump() for conj in all_conjugations], f, indent=2, ensure_ascii=False)

print(f"\nResults saved to {output_file}")

# Retry failed batches if any
if failed_batches:
    print("\n=== Retrying Failed Batches ===")
    retry_conjugations = []
    still_failed = []
    
    for batch in failed_batches:
        print(f"Retrying: {batch}")
        conjugations = get_verb_conjugations(batch)
        
        if conjugations:
            retry_conjugations.extend(conjugations)
        else:
            still_failed.append(batch)
        
        time.sleep(2)  # Longer delay for retries
    
    all_conjugations.extend(retry_conjugations)
    
    # Save updated results
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump([conj.model_dump() for conj in all_conjugations], f, indent=2, ensure_ascii=False)
    
    print(f"\nAfter retry - Total conjugations: {len(all_conjugations)}")
    if still_failed:
        print(f"Still failed batches: {len(still_failed)}")
        print("Failed verbs:")
        for batch in still_failed:
            print(f"  - {batch}")   
